In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
np.set_printoptions(precision=8, suppress=True)

In [2]:
import tensorflow as tf

In [3]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

# The Original Keras' MobileNet Classifer
---

The built-in MobileNet classifier is pre-trained on ImageNet dataset

In [4]:
model = tf.keras.applications.MobileNet(
    input_shape=(224, 224, 3),
    include_top=True,
    weights="imagenet"
)

In [5]:
from tensorflow.keras.utils import load_img, img_to_array

img = load_img("./images/dog.jpeg", target_size=(224, 224))
img = img_to_array(img)

# Add batch dimension
img = tf.expand_dims(img, axis=0)

# MobileNet-specific preprocessing
img = tf.keras.applications.mobilenet.preprocess_input(img)

In [6]:
predictions = model.predict(img)

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


In [7]:
decoded = tf.keras.applications.mobilenet.decode_predictions(
    predictions,
    top=5
)

for _, label, probability in decoded[0]:
    print(label, probability)

golden_retriever 0.75183225
Labrador_retriever 0.15639722
rapeseed 0.052122694
cocker_spaniel 0.009472207
kuvasz 0.0070938417


# My MobileNet
---

I switch out the FC layer and Softmax layer from the original MobileNet Classifier.
I use only the pre-trained MobileNet backbone for my fine-tuning on CIFAR-10 dataset.

To use only the ImageNet-pretrained MobileNet backbone, set `include_top=False`

In [8]:
import tensorflow as tf

num_classes = 10

# Load MobileNet without the original classification head
base_model = tf.keras.applications.MobileNet(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze the backbone (optional for transfer learning)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))

# MobileNet feature extractor
x = base_model(inputs, training=False)

# Convert feature maps into a feature vector
x = tf.keras.layers.GlobalAveragePooling2D()(x)

# Fully connected layer
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)

# Softmax classifier
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_1.00_224 (Functional) │ (None, 7, 7, 1024)     │     3,228,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,361,354 (12.82 MB)

 Trainable params: 132,490 (517.54 KB)

 Non-trainable params: 3,228,864 (12.32 MB)

In [9]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# CIFAR-10
---

In [23]:
import pickle
import numpy as np

def load_batch(filename):
    with open(filename, 'rb') as f:
        data = pickle.load(f, encoding='bytes')

    images = data[b'data']
    labels = data[b'labels']

    images = images.reshape(-1, 3, 32, 32)
    images = images.transpose(0, 2, 3, 1)

    return images, np.array(labels)

# Load one training batch
x_train, y_train = load_batch("cifar-10-batches-py/data_batch_1")

In [24]:
x_train = []
y_train = []

for i in range(1, 6):
    images, labels = load_batch(
        f"cifar-10-batches-py/data_batch_{i}"
    )
    x_train.append(images)
    y_train.append(labels)

x_train = np.concatenate(x_train)
y_train = np.concatenate(y_train)

x_test, y_test = load_batch(
    "cifar-10-batches-py/test_batch"
)

In [25]:
x_train.shape, y_train[:10]

((50000, 32, 32, 3), array([6, 9, 9, 4, 1, 1, 2, 7, 8, 3]))

In [26]:
type(x_train), type(y_train), x_test.shape, y_test[:10]

(numpy.ndarray,
 numpy.ndarray,
 (10000, 32, 32, 3),
 array([3, 8, 8, 0, 6, 6, 1, 6, 3, 1]))

# CIFAR-100
---

In [14]:
import pickle

In [15]:
file = './cifar-100-python/train'
with open(file, 'rb') as fo:
    dict = pickle.load(fo, encoding='bytes')

In [16]:
dict.keys(), dict[b'data'].shape, dict[b'filenames'][:10], dict[b'fine_labels'][:10], dict[b'coarse_labels'][:10]

(dict_keys([b'filenames', b'batch_label', b'fine_labels', b'coarse_labels', b'data']),
 (50000, 3072),
 [b'bos_taurus_s_000507.png',
  b'stegosaurus_s_000125.png',
  b'mcintosh_s_000643.png',
  b'altar_boy_s_001435.png',
  b'cichlid_s_000031.png',
  b'phone_s_002161.png',
  b'car_train_s_000043.png',
  b'beaker_s_000604.png',
  b'fog_s_000397.png',
  b'rogue_elephant_s_000421.png'],
 [19, 29, 0, 11, 1, 86, 90, 28, 23, 31],
 [11, 15, 4, 14, 1, 5, 18, 3, 10, 11])

In [17]:
x_train = dict[b'data']
y_train = dict[b'fine_labels']

In [18]:
file = "./cifar-100-python/test"
with open(file, 'rb') as fo:
    dict = pickle.load(fo, encoding='bytes')

In [19]:
dict.keys(), dict[b'data'].shape, dict[b'filenames'][:10], dict[b'fine_labels'][:10], dict[b'coarse_labels'][:10]

(dict_keys([b'filenames', b'batch_label', b'fine_labels', b'coarse_labels', b'data']),
 (10000, 3072),
 [b'volcano_s_000012.png',
  b'woods_s_000412.png',
  b'seal_s_001803.png',
  b'mushroom_s_001755.png',
  b'adriatic_sea_s_000653.png',
  b'tulipa_clusiana_s_000175.png',
  b'camel_s_001052.png',
  b'mourning_cloak_s_000143.png',
  b'cirrostratus_s_000223.png',
  b'eating_apple_s_000763.png'],
 [49, 33, 72, 51, 71, 92, 15, 14, 23, 0],
 [10, 10, 0, 4, 10, 2, 11, 7, 10, 4])

In [20]:
x_test = dict[b'data']
y_test = dict[b'fine_labels']

In [21]:
# (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Convert labels from shape (N,1) to (N,)
# y_train = y_train.squeeze()
# y_test = y_test.squeeze()

print(x_train.shape)
# (50000, 32, 32, 3)

(50000, 3072)


# Fine-tuning
---

In [27]:
num_classes = 10
img_size = 224
batch_size = 64

def preprocess(image, label):
    image = tf.image.resize(image, (img_size, img_size))
    image = tf.keras.applications.mobilenet.preprocess_input(image)
    return image, label


train_dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (x_test, y_test)
)

train_dataset = (
    train_dataset
    # .shuffle(10000)
    .map(preprocess)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .map(preprocess)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [28]:
base_model.trainable = True

# Freeze the earlier layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath="checkpoints/model_{epoch:02d}.keras",
    save_weights_only=False,   # saves architecture + weights + optimizer state
    save_freq="epoch"
)

In [30]:
model.fit(
    train_dataset,
    epochs=10,
    callbacks=[checkpoint_callback],
    validation_data=test_dataset
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.7047 - loss: 0.8728 - val_accuracy: 0.8254 - val_loss: 0.5503
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - accuracy: 0.7777 - loss: 0.6698 - val_accuracy: 0.8485 - val_loss: 0.4551
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - accuracy: 0.8161 - loss: 0.5623 - val_accuracy: 0.8649 - val_loss: 0.4039
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - accuracy: 0.8357 - loss: 0.4923 - val_accuracy: 0.8739 - val_loss: 0.3706
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - accuracy: 0.8522 - loss: 0.4449 - val_accuracy: 0.8827 - val_loss: 0.3473
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - accuracy: 0.8652 - loss: 0.4055 - val_accuracy: 0.8881 - val_loss: 0.3282
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - accuracy: 0.8772 - loss: 0.3713 - val_accuracy: 0.8927 - val_loss: 0.3142
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - accuracy: 0.8862 - loss: 0.3430 - 

## Interrupt and Restart Training

In [34]:
model = tf.keras.models.load_model(
    "checkpoints/model_10.keras"
)

In [35]:
model.fit(
    train_dataset,
    initial_epoch= 10,
    epochs=10,
    callbacks=[checkpoint_callback],
)

## Evaluate

In [78]:
img = load_img("./images/cat.jpeg", target_size=(224, 224))
img = img_to_array(img)

# Add batch dimension
img = tf.expand_dims(img, axis=0)

# MobileNet-specific preprocessing
img = tf.keras.applications.mobilenet.preprocess_input(img)

In [79]:
predictions = model.predict(img)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


In [83]:
predictions

array([[0.00003114, 0.00000101, 0.00008343, 0.9997162 , 0.00009788,
        0.00002609, 0.00003868, 0.00000366, 0.00000156, 0.00000033]],
      dtype=float32)

The fourth class is predicted high probability and the fourth class represents cat, so the prediction is correct.

In [ ]:
decoded = tf.keras.applications.mobilenet.decode_predictions(
    predictions,
    top=5
)

for _, label, probability in decoded[0]:
    print(label, probability)